# 1. Evaluating GPT

## 1.1 Generating Text

In [1]:
import torch
from gpt_model import GPTModel

In [5]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,   # Vocabulary size
    "context_length": 256, # Shortened context length (orig: 1024)
    "emb_dim": 768,        # Embedding dimension
    "n_heads": 12,         # Number of attention heads
    "n_layers": 12,        # Number of layers
    "drop_rate": 0.1,      # Dropout rate
    "qkv_bias": False,      # Query-key-value bias
    "weight_tying": False
}

In [6]:
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval();  # Disable dropout during inference

![](images/model-training-1.png)

### Helper Functions for Encoding and Decoding Text

In [7]:
import tiktoken
from generate_text import generate_text_simple

def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    # add batch dimension
    encoded_tensor = torch.tensor(encoded).unsqueeze(0)
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    # remove batch dimension
    flat = token_ids.squeeze(0)
    return tokenizer.decode(flat.tolist())

In [11]:
start_context = "Every effort moves you"
tokenizer = tiktoken.get_encoding("gpt2")

token_ids = generate_text_simple(
    model=model,
    current_context=text_to_token_ids(start_context, tokenizer),
    max_new_tokens=10,
    context_size=GPT_CONFIG_124M["context_length"]
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

Output text:
 Every effort moves you rentingetic wasnم refres RexMeCHicular stren


## 1.2 Calculating the quality of the generation - cross-entropy and perplexity

In [13]:
# Define inputs and the corresponding targets

inputs = torch.tensor([[16833, 3626, 6100],   # ["every effort moves",
                       [40,    1107, 588]])   #  "I really like"]

targets = torch.tensor([[3626, 6100, 345  ],  # [" effort moves you",
                        [1107,  588, 11311]]) #  " really like chocolate"]

In [14]:
# Pass the inputs through the model and get the predictions

with torch.no_grad():
    logits = model(inputs)

# logits will be of the form: [batch_size, num_tokens, vocab_size]
# i.e., [2, 3, 50257]
# so, we compute the softmax for each token position along the last dimension
probas = torch.softmax(logits, dim=-1)

print(probas.shape)

torch.Size([2, 3, 50257])


Converting the probability scores back into text

![](images/model-training-2.png)

In [18]:
# Pick the token with the max probability as the "predicted" outputs

predicted_token_ids = torch.argmax(probas, dim=-1, keepdim=True)
print("Token IDs:\n", predicted_token_ids)

Token IDs:
 tensor([[[16657],
         [  339],
         [42826]],

        [[49906],
         [29669],
         [41751]]])


In [17]:
# Compare the actual vs predicted outputs

print(f"Targets batch 1: {token_ids_to_text(targets[0], tokenizer)}")
print(f"Outputs batch 1: {token_ids_to_text(predicted_token_ids[0].flatten(), tokenizer)}")

print(f"Targets batch 2: {token_ids_to_text(targets[1], tokenizer)}")
print(f"Outputs batch 2: {token_ids_to_text(predicted_token_ids[1].flatten(), tokenizer)}")

Targets batch 1:  effort moves you
Outputs batch 1:  Armed heNetflix
Targets batch 2:  really like chocolate
Outputs batch 2:  pressuring empoweredfaith


In [22]:
# Inspect the probabilities for the actual target tokens

# Gimme the probabilities for token IDs targets[0][0], 
# targets[0][1] and targets[0][2] at index 0, 1 and 2
# respectively
target_probas_1 = probas[0, [0, 1, 2], targets[0]]
print("Text 1:", target_probas_1)

# Gimme the probabilities for token IDs targets[1][0], 
# targets[1][1] and targets[1][2] at index 0, 1 and 2
# respectively
target_probas_2 = probas[1, [0, 1, 2], targets[1]]
print("Text 2:", target_probas_2)

Text 1: tensor([7.4541e-05, 3.1061e-05, 1.1563e-05])
Text 2: tensor([1.0337e-05, 5.6776e-05, 4.7559e-06])


We wanna maximize the above values, which would mean the model very strongly predicted what we actually wanted it to predict!

In [23]:
# Compute the log loss across all tokens
log_probas = torch.log(torch.cat((target_probas_1, target_probas_2)))
print(log_probas)

tensor([ -9.5042, -10.3796, -11.3677, -11.4798,  -9.7764, -12.2561])


In [24]:
# Average it out
avg_log_probas = torch.mean(log_probas)
print(avg_log_probas)

tensor(-10.7940)


We wanna maximize this value! Since it's a log, the max value is 0

In [25]:
# As per ML convention, minimize the average log probability

neg_avg_log_probas = avg_log_probas * -1
print(neg_avg_log_probas)

tensor(10.7940)


### Cross-entropy

PyTorch already implements a cross_entropy function that carries out the previous steps

Cross-entropy calculates how far two probability distributions are from each other.

**Lower is better**

![](images/model-training-3.png)

In [26]:
# Logits have shape (batch_size, num_tokens, vocab_size)
print("Logits shape:", logits.shape)

# Targets have shape (batch_size, num_tokens)
print("Targets shape:", targets.shape)

Logits shape: torch.Size([2, 3, 50257])
Targets shape: torch.Size([2, 3])


In [27]:
# For the cross_entropy function in PyTorch, we want to flatten these tensors 
# by combining them over the batch dimension

logits_flat = logits.flatten(0, 1)
targets_flat = targets.flatten()

print("Flattened logits:", logits_flat.shape)
print("Flattened targets:", targets_flat.shape)

Flattened logits: torch.Size([6, 50257])
Flattened targets: torch.Size([6])


In [28]:
loss = torch.nn.functional.cross_entropy(logits_flat, targets_flat)
print(loss)

tensor(10.7940)


* The `cross_entropy` function knows the targe token IDs we ideally wanted (the second param). 
* It then looks at the logits for each of the 6 tokens and inspects the compute probs for these token IDs. 
* It then gives us the loss value, which represents how far away the model was from predicting really high values
corresponding to the tokens that we actually wanted.

### Perplexity

Related term which measures how well the probability distribution predicted by the model matches the actual distribution of the words in the dataset.

**Example: if perplexity is 48725, then it means that at each step, that the model picked the next token as if picking randomly from 48725 options.**

**It is simply = e^cross-entropy**

**Lower is better.**

In [29]:
perplexity = torch.exp(loss)
print(perplexity)

tensor(48725.8203)
